In [1]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

d:\Desktop\ai-projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4188.77it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight      

In [2]:
import torch

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = model(**inputs)
    logits = outputs.logits

    predicted_class = torch.argmax(logits, dim=1).item()

    return predicted_class

In [3]:
print(predict_sentiment("Doctor was very kind and helpful"))
print(predict_sentiment("Very bad service, I am angry"))

1
1


In [4]:
!pip install datasets

from datasets import load_dataset

dataset = load_dataset("imdb")

In [6]:
dataset["train"]
dataset["test"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [11]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [ ]:
print(dataset["train"]["label"][:500
])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [20]:
import collections

labels = dataset["train"]["label"][:1000]
print(collections.Counter(labels))

Counter({0: 1000})


In [21]:
train_data = dataset["train"].shuffle(seed=42)

In [22]:
labels = train_data["label"][:1000]

import collections
print(collections.Counter(labels))

Counter({0: 512, 1: 488})


In [23]:
train_data = dataset["train"].shuffle(seed=42).select(range(2000))

In [26]:
texts = list(train_data["text"])
labels = list(train_data["label"])

In [27]:
encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [29]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [35]:
import torch

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [36]:
train_dataset = ReviewDataset(encodings, labels)

In [37]:
print(len(train_dataset))

2000


In [38]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [39]:
from tqdm import tqdm

model.train()

for epoch in range(2):
    print(f"Epoch {epoch+1}")

    loop = tqdm(train_loader)

    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=loss.item())

Epoch 1


100%|██████████| 250/250 [27:21<00:00,  6.57s/it, loss=0.723] 


Epoch 2


100%|██████████| 250/250 [29:34<00:00,  7.10s/it, loss=0.0927]


In [40]:
model.save_pretrained("sentiment-model")
tokenizer.save_pretrained("sentiment-model")

Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.33s/it]


('sentiment-model\\tokenizer_config.json', 'sentiment-model\\tokenizer.json')

In [43]:
print(predict_sentiment("doctor was good but waiting time was too much"))
print(predict_sentiment("Very bad service, I am angry"))

0
0


In [44]:
print(predict_sentiment("Doctor was very kind and helpful"))

1


In [46]:
print(predict_sentiment("waiting time also very high "))

1


In [47]:
texts = [
    "doctor was very kind and helpful",
    "waiting time was too long",
    "food was delicious and fresh",
    "service was very slow",
    "staff was polite and friendly",
    "restaurant was not clean",
    "cafe ambiance was amazing",
    "order took too much time"
]

labels = [1, 0, 1, 0, 1, 0, 1, 0]

In [48]:
encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [49]:
fine_dataset = ReviewDataset(encodings, labels)

In [50]:
from torch.utils.data import DataLoader

fine_loader = DataLoader(fine_dataset, batch_size=4, shuffle=True)

In [51]:
model.train()

for epoch in range(4):   # small data → more epochs
    print(f"Fine-tune Epoch {epoch+1}")

    for batch in fine_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

    print("loss:", loss.item())

Fine-tune Epoch 1
loss: 0.1322111189365387
Fine-tune Epoch 2
loss: 0.11287379264831543
Fine-tune Epoch 3
loss: 0.04570872336626053
Fine-tune Epoch 4
loss: 0.02826470509171486


In [52]:
print(predict_sentiment("waiting time was very high"))
print(predict_sentiment("service was very slow"))
print(predict_sentiment("doctor was very kind"))
print(predict_sentiment("food was amazing"))
print(predict_sentiment("staff was rude"))

0
0
1
1
0


In [53]:
model.save_pretrained("final-model")
tokenizer.save_pretrained("final-model")

Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.34s/it]


('final-model\\tokenizer_config.json', 'final-model\\tokenizer.json')

In [54]:
def get_reply(review):
    sentiment = predict_sentiment(review)

    if sentiment == 0:
        return "We’re sorry for your experience. We will improve our service."
    else:
        return "Thank you for your valuable feedback!"

In [55]:
test_reviews = [
    "waiting time was very high",
    "doctor was very helpful",
    "service was too slow",
    "food was delicious",
    "staff was rude",
    "very clean and hygienic place"
]

for r in test_reviews:
    print(r, "→", get_reply(r))

waiting time was very high → We’re sorry for your experience. We will improve our service.
doctor was very helpful → Thank you for your valuable feedback!
service was too slow → We’re sorry for your experience. We will improve our service.
food was delicious → Thank you for your valuable feedback!
staff was rude → We’re sorry for your experience. We will improve our service.
very clean and hygienic place → Thank you for your valuable feedback!
